# Aim 2: data download and checks

This notebook only gets and checks data. Analysis is in `aim2_shared_well_effect.ipynb`.

1. **Parse 10M PBMC cytokines**: already downloaded for Aim 1. The cells below check the file and resume the download if it is missing.
2. **Tahoe-100M metadata only** (not the 429 GB of counts): to check whether every sample has a plate and well coordinate, and where the DMSO wells sit.

In [29]:
%pip install -q anndata h5py huggingface_hub pandas pyarrow


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
import os, subprocess
import pandas as pd

PARSE_DIR = "data/parse10m"
PARSE_H5AD = os.path.join(PARSE_DIR, "20260203_Parse_10M_PBMC_cytokines.h5ad")
PARSE_ZIP_URL = "https://parse-wget.s3.us-west-2.amazonaws.com/10m/10M_PBMC_12donor_90cytokines_h5ad_20260203.zip"

TAHOE_DIR = "data/tahoe100m_metadata"
TAHOE_REPO = "tahoebio/Tahoe-100M"

OUT_DIR = "results/cache/aim2_download"
for d in [PARSE_DIR, TAHOE_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

## 1. Parse 10M

If the `.h5ad` file already exists, nothing is downloaded. Otherwise the zip is downloaded (resumable, about 2.5 h) and extracted.

In [31]:
import os, glob, shutil

OLD_DIR = os.path.abspath("../Proposal2/data/parse10m")   # where Parse 10M already is
NEW_DIR = os.path.abspath("data/parse10m")
FNAME = "20260203_Parse_10M_PBMC_cytokines.h5ad"
os.makedirs(NEW_DIR, exist_ok=True)

# 1) Remove the partial download started in Proposal3
for f in glob.glob(os.path.join(NEW_DIR, "*.zip")):
    print("Removing partial file:", f, f"({os.path.getsize(f)/1e9:.1f} GB)")
    os.remove(f)

# 2) Copy the h5ad (skipped if a complete copy is already here)
src, dst = os.path.join(OLD_DIR, FNAME), os.path.join(NEW_DIR, FNAME)
if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
    print("Already copied.")
else:
    print(f"Copying {os.path.getsize(src)/1e9:.1f} GB ...")
    shutil.copy2(src, dst)

# 3) Check
same = os.path.getsize(src) == os.path.getsize(dst)
print("Size match:", same, f"({os.path.getsize(dst)/1e9:.1f} GB)")
print(os.listdir(NEW_DIR))

Already copied.
Size match: True (44.4 GB)
['20260203_Parse_10M_PBMC_cytokines.h5ad']


In [32]:
if os.path.exists(PARSE_H5AD):
    print(f"Found: {PARSE_H5AD}  ({os.path.getsize(PARSE_H5AD) / 1e9:.1f} GB)")
else:
    zip_path = os.path.join(PARSE_DIR, os.path.basename(PARSE_ZIP_URL))
    print("Downloading Parse 10M (resumable) ...")
    subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", zip_path, PARSE_ZIP_URL], check=True)
    subprocess.run(["unzip", "-n", "-q", zip_path, "-d", PARSE_DIR], check=True)
    print("Done. Files:", os.listdir(PARSE_DIR))

Found: data/parse10m/20260203_Parse_10M_PBMC_cytokines.h5ad  (44.4 GB)


In [33]:
import anndata as ad

a = ad.read_h5ad(PARSE_H5AD, backed="r")
print(f"{a.n_obs:,} cells x {a.n_vars:,} genes")
print("obs columns:", a.obs.columns.tolist())
need = ["donor", "bc1_well", "cytokine", "cell_type"]
missing = [c for c in need if c not in a.obs.columns]
print("Required columns present:", not missing, "| missing:", missing)

wells = a.obs[["donor", "bc1_well", "cytokine"]].drop_duplicates()
print("Wells per donor:", wells.groupby("donor", observed=True).size().to_dict())
pbs = wells[wells.cytokine.astype(str) == "PBS"]
print("PBS wells (first donor):", sorted(pbs[pbs.donor == pbs.donor.iloc[0]].bc1_well.astype(str)))
print("Fine cell types:", sorted(a.obs.cell_type.astype(str).unique()))
a.file.close()

9,697,974 cells x 40,352 genes
obs columns: ['sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type']
Required columns present: True | missing: []
Wells per donor: {'Donor1': 96, 'Donor2': 96, 'Donor3': 96, 'Donor4': 96, 'Donor5': 96, 'Donor6': 96, 'Donor7': 96, 'Donor8': 96, 'Donor9': 96, 'Donor10': 96, 'Donor11': 96, 'Donor12': 96}
PBS wells (first donor): ['H10', 'H11', 'H12', 'H7', 'H8', 'H9']
Fine cell types: ['B Intermediate/Memory', 'B Naive', 'CD14 Mono', 'CD16 Mono', 'CD4 Memory', 'CD4 Naive', 'CD8 Memory', 'CD8 Naive', 'HSPC', 'ILC', 'MAIT', 'NK', 'NK CD56bright', 'NKT', 'Plasmablast', 'Treg', 'cDC', 'pDC']


## 2. Tahoe-100M metadata

Only the small files in the `metadata/` folder of the Hugging Face repo are downloaded. The cell lists the repo first, so it works even if file names change.

In [34]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # progress widgets can freeze in Jupyter
from huggingface_hub import list_repo_files, hf_hub_download

files = list_repo_files(TAHOE_REPO, repo_type="dataset")
# Only the small sample-level tables. Skip obs_metadata (one row per cell, very large),
# gene files, and the 1,026 pseudobulk DE files.
WANTED = ["metadata/sample_metadata.parquet",
          "metadata/drug_metadata.parquet",
          "metadata/cell_line_metadata.parquet"]
meta_files = [f for f in WANTED if f in files]
print("Downloading:", meta_files)

local = {}
for f in meta_files:
    local[f] = hf_hub_download(TAHOE_REPO, f, repo_type="dataset", local_dir=TAHOE_DIR)
    print(f"  {f}: {os.path.getsize(local[f]) / 1e6:.2f} MB")
print("Saved to", TAHOE_DIR)

Downloading: ['metadata/sample_metadata.parquet', 'metadata/drug_metadata.parquet', 'metadata/cell_line_metadata.parquet']
  metadata/sample_metadata.parquet: 0.07 MB
  metadata/drug_metadata.parquet: 0.04 MB
  metadata/cell_line_metadata.parquet: 0.02 MB
Saved to data/tahoe100m_metadata


In [35]:
# Show the columns of every metadata table and look for plate / well columns
tables = {}
for f, p in local.items():
    if p.endswith(".parquet"):
        tables[f] = pd.read_parquet(p)
    elif p.endswith(".csv"):
        tables[f] = pd.read_csv(p)
for f, t in tables.items():
    print(f"\n=== {f}  shape={t.shape}")
    print("columns:", t.columns.tolist())
    print(t.head(3).to_string())

well_like = {f: [c for c in t.columns if any(k in c.lower() for k in ["well", "plate", "row", "col"])]
             for f, t in tables.items()}
print("\nPlate/well-like columns per table:", well_like)


=== metadata/sample_metadata.parquet  shape=(1344, 8)
columns: ['sample', 'plate', 'mean_gene_count', 'mean_tscp_count', 'mean_mread_count', 'mean_pcnt_mito', 'drug', 'drugname_drugconc']
     sample   plate  mean_gene_count  mean_tscp_count  mean_mread_count  mean_pcnt_mito          drug               drugname_drugconc
0  smp_1495  plate1      1354.169768      2027.115940       2444.032416        0.033956  Infigratinib  [('Infigratinib', 0.05, 'uM')]
1  smp_1496  plate1      1404.454157      2226.282791       2690.685970        0.071723  Erdafitinib   [('Erdafitinib ', 0.05, 'uM')]
2  smp_1497  plate1      1205.267794      1859.375821       2246.200127        0.084853    Everolimus    [('Everolimus', 0.05, 'uM')]

=== metadata/drug_metadata.parquet  shape=(379, 9)
columns: ['drug', 'targets', 'moa-broad', 'moa-fine', 'human-approved', 'clinical-trials', 'gpt-notes-approval', 'canonical_smiles', 'pubchem_cid']
         drug targets             moa-broad              moa-fine human-app

In [36]:
# If a table has plate and well columns, summarise the layout and the DMSO wells.
summary = []
for f, t in tables.items():
    cols = [c.lower() for c in t.columns]
    plate_c = next((c for c in t.columns if "plate" in c.lower()), None)
    well_c = next((c for c in t.columns if c.lower() == "well" or "well" in c.lower()), None)
    drug_c = next((c for c in t.columns if c.lower() in ["drug", "drugname", "drug_name", "sm_name", "compound"]), None)
    if plate_c and well_c:
        n = t.groupby(plate_c)[well_c].nunique()
        print(f"\n{f}: wells per plate:\n{n.to_string()}")
        if drug_c:
            ctrl = t[t[drug_c].astype(str).str.contains("DMSO", case=False)]
            pos = ctrl.groupby(plate_c)[well_c].apply(lambda s: sorted(s.astype(str).unique()))
            print(f"\nDMSO well positions per plate:\n{pos.to_string()}")
            summary.append(pos.rename("dmso_wells").reset_index())
if summary:
    pd.concat(summary).to_csv(os.path.join(OUT_DIR, "tahoe_dmso_wells.csv"), index=False)
    print("\nSaved", os.path.join(OUT_DIR, "tahoe_dmso_wells.csv"))
else:
    print("\nNo table with both plate and well columns. Send me the printed columns above.")


No table with both plate and well columns. Send me the printed columns above.


## 3. Tahoe: infer well positions from sample order

`sample_metadata` has `plate` but no `well` column. The paper says DMSO controls are in wells **H11 and H12** of every plate. If sample IDs were assigned in well order, the DMSO samples should sit at the **same positions** in every plate, and those positions tell us whether the order is row-wise (A1, A2, ..., H12) or column-wise (A1, B1, ..., H12).

This is an **inference**, not the official layout. If it holds, we still ask the authors to confirm.

In [37]:
sm = tables["metadata/sample_metadata.parquet"].copy()
sm["smp_num"] = sm["sample"].str.replace("smp_", "", regex=False).astype(int)
sm = sm.sort_values(["plate", "smp_num"])
sm["pos"] = sm.groupby("plate").cumcount()               # 0..95 within each plate
sm["is_dmso"] = sm["drug"].astype(str).str.strip() == "DMSO_TF"   # exact match; "Trametinib (DMSO_TF solvate)" is a drug

print("Samples per plate:", sm.groupby("plate").size().to_dict())
print("DMSO samples per plate:", sm.groupby("plate").is_dmso.sum().to_dict())
print("Are sample numbers consecutive within each plate?",
      sm.groupby("plate").smp_num.apply(lambda s: (s.diff().dropna() == 1).all()).to_dict())

dmso_pos = sm[sm.is_dmso].groupby("plate").pos.apply(list)
print("\nDMSO positions (0-based) per plate:\n", dmso_pos.to_string())

# Expected positions of H11 and H12 under two orders
rows = "ABCDEFGH"
row_major = {f"{rows[i // 12]}{i % 12 + 1}": i for i in range(96)}
col_major = {f"{rows[i % 8]}{i // 8 + 1}": i for i in range(96)}
print("\nExpected H11, H12 positions | row-wise:", [row_major["H11"], row_major["H12"]],
      "| column-wise:", [col_major["H11"], col_major["H12"]])

same_everywhere = dmso_pos.apply(tuple).nunique() == 1
print("DMSO at the same positions in every plate:", same_everywhere)

Samples per plate: {'plate1': 96, 'plate10': 96, 'plate11': 96, 'plate12': 96, 'plate13': 96, 'plate14': 96, 'plate2': 96, 'plate3': 96, 'plate4': 96, 'plate5': 96, 'plate6': 96, 'plate7': 96, 'plate8': 96, 'plate9': 96}
DMSO samples per plate: {'plate1': 2, 'plate10': 2, 'plate11': 2, 'plate12': 2, 'plate13': 2, 'plate14': 2, 'plate2': 2, 'plate3': 2, 'plate4': 2, 'plate5': 2, 'plate6': 2, 'plate7': 2, 'plate8': 2, 'plate9': 2}
Are sample numbers consecutive within each plate? {'plate1': True, 'plate10': True, 'plate11': True, 'plate12': True, 'plate13': True, 'plate14': True, 'plate2': True, 'plate3': True, 'plate4': True, 'plate5': True, 'plate6': True, 'plate7': True, 'plate8': True, 'plate9': True}

DMSO positions (0-based) per plate:
 plate
plate1     [94, 95]
plate10    [94, 95]
plate11    [94, 95]
plate12    [94, 95]
plate13    [22, 23]
plate14    [94, 95]
plate2     [94, 95]
plate3     [94, 95]
plate4     [94, 95]
plate5     [94, 95]
plate6     [94, 95]
plate7     [94, 95]
pla

In [38]:
# Row-wise order fits most plates (DMSO at positions 94, 95 = H11, H12).
# Plates that do not fit are kept but flagged, so they can be left out of well-level analyses.
inv = {v: k for k, v in row_major.items()}
H11_H12 = {row_major["H11"], row_major["H12"]}

def fits(positions):
    return H11_H12.issubset(set(positions))

plate_ok = dmso_pos.apply(fits)
print("Plates where H11/H12 = DMSO under row-wise order:", int(plate_ok.sum()), "of", len(plate_ok))
print("Plates that do not fit:", plate_ok[~plate_ok].index.tolist())

# DMSO samples outside H11/H12 (plate13: B11/B12 under row-wise order)
extra = sm[sm.is_dmso & ~sm.pos.isin(H11_H12)]
print("\nDMSO samples outside H11/H12:")
print(extra[["plate", "sample", "pos", "drug", "drugname_drugconc"]].to_string(index=False))

sm["well"] = sm.pos.map(inv)
sm["row"] = sm.well.str[0]
sm["col"] = sm.well.str[1:].astype(int)
sm["well_confident"] = sm.plate.map(plate_ok)
sm["well_source"] = "inferred: row-wise sample order, DMSO at H11/H12"

out = os.path.join(OUT_DIR, "tahoe_sample_wells_inferred.csv")
sm.drop(columns=["drugname_drugconc"]).to_csv(out, index=False)
print("\nSaved", out)
print("\nLayout of plate6 (inferred):")
print(sm[sm.plate == "plate6"].pivot(index="row", columns="col", values="drug").to_string())

Plates where H11/H12 = DMSO under row-wise order: 13 of 14
Plates that do not fit: ['plate13']

DMSO samples outside H11/H12:
  plate   sample  pos    drug        drugname_drugconc
plate13 smp_2669   22 DMSO_TF [('DMSO_TF', 0.0, 'uM')]
plate13 smp_2670   23 DMSO_TF [('DMSO_TF', 0.0, 'uM')]

Saved results/cache/aim2_download/tahoe_sample_wells_inferred.csv

Layout of plate6 (inferred):
col                          1                          2                  3                               4                              5                           6                             7                               8                          9                             10                                            11                                       12
row                                                                                                                                                                                                                                          

## 4. Tahoe: per-plate h5ad files (plates 6, 14, 13)

From the Arc Virtual Cell Atlas public bucket. Plates 6 and 14 are replicates; plate 13 has DMSO in row B. About 75 GB in total, resumable.

In [39]:
import json, subprocess, urllib.request

TAHOE_H5AD_DIR = "data/tahoe100m_h5ad"
BUCKET = "arc-institute-virtual-cell-atlas"
PREFIX = "tahoe100M/2025-02-25/h5ad/"
PLATES = ["plate6", "plate14", "plate13"]
os.makedirs(TAHOE_H5AD_DIR, exist_ok=True)

# List the bucket to get the exact file names and sizes
url = f"https://storage.googleapis.com/storage/v1/b/{BUCKET}/o?prefix={PREFIX}&fields=items(name,size)"
items = json.load(urllib.request.urlopen(url))["items"]
files = {it["name"].split("/")[-1]: int(it["size"]) for it in items if it["name"].endswith(".h5ad")}
todo = {f: s for f, s in files.items() if any(f.startswith(p + "_") for p in PLATES)}
for f, s in todo.items():
    print(f"{s / 1e9:6.1f} GB  {f}")
print(f"Total: {sum(todo.values()) / 1e9:.1f} GB")

  28.1 GB  plate13_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad
  22.4 GB  plate14_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad
  28.9 GB  plate6_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad
Total: 79.4 GB


In [41]:
import time
for f, size in todo.items():
    out = os.path.join(TAHOE_H5AD_DIR, f)
    if os.path.exists(out) and os.path.getsize(out) == size:
        print("Already complete:", f)
        continue
    t0 = time.time()
    print(f"Downloading {f} ({size / 1e9:.1f} GB) ...", flush=True)
    subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", out,
                    f"https://storage.googleapis.com/{BUCKET}/{PREFIX}{f}"], check=True)
    ok = os.path.getsize(out) == size
    print(f"  done in {(time.time() - t0) / 60:.0f} min | size match: {ok}", flush=True)

Already complete: plate13_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad
Already complete: plate14_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad
Already complete: plate6_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad
